In [ ]:
!pip install pycuda


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 10.4 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5316537 sha256=b8bca410c5f697bf7c2514117f9ea8a4790e1b14db738dd2c337293bc380cb7d
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


In [ ]:
"""Global-memory and shared-memory moving-average stencils with PyCUDA.

For a stencil radius R, the code computes the unnormalized moving sum

    y[i] = sum_{r=-R}^{R} x[i+r],

with zero padding outside the array.  Dividing by 2*R+1 would give the usual
moving average and would affect both kernels in exactly the same way, so the
sum is sufficient for a memory-reuse benchmark.

The key experiment varies R while keeping the array size and block size fixed.
As R grows, each input value contributes to more neighboring outputs.  The
shared-memory implementation loads a block tile plus two halos once and reuses
those values for all stencil evaluations in the block.

A CUDA-capable NVIDIA GPU, PyCUDA, and a working CUDA installation are required.
"""

In [ ]:
from __future__ import annotations
import argparse
from dataclasses import dataclass
import numpy as np

In [ ]:
try:
    import pycuda.autoinit  # noqa: F401
    import pycuda.driver as cuda
    from pycuda.compiler import SourceModule
    import pycuda.gpuarray as gpuarray
except ImportError as exc:  # pragma: no cover - requires CUDA
    raise SystemExit(
        "PyCUDA and a working NVIDIA CUDA installation are required."
    ) from exc


In [ ]:
CUDA_SOURCE = r"""
extern "C" {

__global__ void moving_sum_global(
    const float * __restrict__ input,
    float * __restrict__ output,
    const int n,
    const int radius)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    float sum = 0.0f;
    for (int offset = -radius; offset <= radius; ++offset) {
        const int j = i + offset;
        if (j >= 0 && j < n) {
            sum += input[j];
        }
    }
    output[i] = sum;
}

__global__ void moving_sum_shared(
    const float * __restrict__ input,
    float * __restrict__ output,
    const int n,
    const int radius)
{
    extern __shared__ float tile[];

    const int tx = threadIdx.x;
    const int block_start = blockIdx.x * blockDim.x;
    const int i = block_start + tx;
    const int local = tx + radius;

    // Central block: one load per thread.
    tile[local] = (i < n) ? input[i] : 0.0f;

    // Halo loading is written as a strided loop so the kernel remains correct
    // even when radius approaches or exceeds the number of threads in a block.
    for (int h = tx; h < radius; h += blockDim.x) {
        const int left = block_start - radius + h;
        const int right = block_start + blockDim.x + h;
        tile[h] = (left >= 0 && left < n) ? input[left] : 0.0f;
        tile[radius + blockDim.x + h] =
            (right >= 0 && right < n) ? input[right] : 0.0f;
    }

    __syncthreads();

    if (i >= n) return;

    float sum = 0.0f;
    for (int offset = -radius; offset <= radius; ++offset) {
        sum += tile[local + offset];
    }
    output[i] = sum;
}

} // extern "C"
"""


In [ ]:
@dataclass
class TimingRow:
    radius: int
    reuse: int
    ideal_global_reads_per_output: float
    staged_global_reads_per_output: float
    global_ms: float
    shared_ms: float
    speedup: float
    max_abs_difference: float


In [ ]:
class MovingAverageKernels:
    """Compile the global and shared moving-sum kernels once."""

    def __init__(self) -> None:
        module = SourceModule(CUDA_SOURCE, options=["-O3"], no_extern_c=True)
        self.global_kernel = module.get_function("moving_sum_global")
        self.shared_kernel = module.get_function("moving_sum_shared")


In [ ]:
def cpu_reference(a: np.ndarray, radius: int) -> np.ndarray:
    """Reference moving sum with zero padding, evaluated efficiently on CPU."""
    if radius < 0:
        raise ValueError("radius must be non-negative")
    # Prefix sums avoid an O(N*R) Python loop and are sufficiently accurate for
    # validation when accumulated in float64.
    padded = np.pad(a.astype(np.float64), (radius, radius), mode="constant")
    prefix = np.empty(padded.size + 1, dtype=np.float64)
    prefix[0] = 0.0
    np.cumsum(padded, dtype=np.float64, out=prefix[1:])
    width = 2 * radius + 1
    return (prefix[width:] - prefix[:-width]).astype(np.float32)


In [ ]:
def _event_time_ms(launch, launches: int) -> float:
    """Return average device time per launch using CUDA events."""
    start = cuda.Event()
    stop = cuda.Event()
    start.record()
    for _ in range(launches):
        launch()
    stop.record()
    stop.synchronize()
    return float(start.time_till(stop)) / launches


In [ ]:
def benchmark_radius(
    a: np.ndarray,
    radius: int,
    *,
    block_size: int,
    warmup: int,
    repetitions: int,
    launches_per_sample: int,
    kernels: MovingAverageKernels,
) -> TimingRow:
    """Validate and benchmark one stencil radius."""
    if radius < 1:
        raise ValueError("radius must be at least 1 for this comparison")
    if block_size <= 0:
        raise ValueError("block_size must be positive")

    n = int(a.size)
    reference = cpu_reference(a, radius)

    a_gpu = gpuarray.to_gpu(a)
    global_gpu = gpuarray.empty_like(a_gpu)
    shared_gpu = gpuarray.empty_like(a_gpu)

    grid_size = (n + block_size - 1) // block_size
    block = (block_size, 1, 1)
    grid = (grid_size, 1, 1)
    shared_bytes = (block_size + 2 * radius) * np.dtype(np.float32).itemsize
    n32 = np.int32(n)
    radius32 = np.int32(radius)

    # Check that the requested dynamic shared-memory allocation fits the GPU.
    device = cuda.Context.get_device()
    max_shared = int(device.get_attribute(cuda.device_attribute.MAX_SHARED_MEMORY_PER_BLOCK))
    if shared_bytes > max_shared:
        raise ValueError(
            f"radius={radius} needs {shared_bytes} B of shared memory, "
            f"but this device allows {max_shared} B per block"
        )

    def launch_global() -> None:
        kernels.global_kernel(
            a_gpu,
            global_gpu,
            n32,
            radius32,
            block=block,
            grid=grid,
        )

    def launch_shared() -> None:
        kernels.shared_kernel(
            a_gpu,
            shared_gpu,
            n32,
            radius32,
            block=block,
            grid=grid,
            shared=shared_bytes,
        )

    for _ in range(warmup):
        launch_global()
        launch_shared()
    cuda.Context.synchronize()

    global_host = global_gpu.get()
    shared_host = shared_gpu.get()

    # Floating-point summation order differs from the prefix-sum CPU reference,
    # so use a scale-aware tolerance rather than bitwise equality.
    scale = max(float(np.max(np.abs(reference))), 1.0)
    error_global = float(np.max(np.abs(global_host - reference))) / scale
    error_shared = float(np.max(np.abs(shared_host - reference))) / scale
    if error_global > 2e-5 or error_shared > 2e-5:
        raise RuntimeError(
            f"validation failed at radius={radius}: "
            f"global={error_global:.3e}, shared={error_shared:.3e}"
        )

    global_samples: list[float] = []
    shared_samples: list[float] = []
    for rep in range(repetitions):
        if rep % 2 == 0:
            global_samples.append(_event_time_ms(launch_global, launches_per_sample))
            shared_samples.append(_event_time_ms(launch_shared, launches_per_sample))
        else:
            shared_samples.append(_event_time_ms(launch_shared, launches_per_sample))
            global_samples.append(_event_time_ms(launch_global, launches_per_sample))

    global_ms = float(np.median(global_samples))
    shared_ms = float(np.median(shared_samples))

    # Ignoring caches and block edges, the direct kernel requests 2R+1 input
    # values per output. The tiled kernel requests roughly B+2R global values
    # for B outputs, i.e. 1+2R/B global loads per output.
    direct_reads = float(2 * radius + 1)
    staged_reads = 1.0 + 2.0 * radius / block_size

    return TimingRow(
        radius=radius,
        reuse=2 * radius + 1,
        ideal_global_reads_per_output=direct_reads,
        staged_global_reads_per_output=staged_reads,
        global_ms=global_ms,
        shared_ms=shared_ms,
        speedup=global_ms / shared_ms,
        max_abs_difference=float(np.max(np.abs(global_host - shared_host))),
    )


In [ ]:
def benchmark_reuse(
    *,
    n: int = 2**22,
    radii: tuple[int, ...] = (1, 2, 4, 8, 16, 32),
    block_size: int = 256,
    warmup: int = 3,
    repetitions: int = 7,
    launches_per_sample: int = 10,
) -> list[TimingRow]:
    """Vary stencil radius to expose the effect of increasing data reuse."""
    if n <= 0:
        raise ValueError("n must be positive")
    if not radii:
        raise ValueError("at least one radius is required")

    # Deterministic signal used by every radius so only stencil width changes.
    x = np.arange(n, dtype=np.float32)
    a = (0.6 * np.sin(0.0007 * x) + 0.4 * np.cos(0.0019 * x)).astype(np.float32)

    kernels = MovingAverageKernels()
    rows: list[TimingRow] = []

    print("\nMoving sum: direct global memory vs shared-memory tiling")
    print("speedup = T_global / T_shared; values > 1 favor shared memory")
    print("reuse = 2R+1 is the number of neighboring outputs using an interior input\n")
    print(
        f"{'R':>4s} {'reuse':>7s} {'global rd/out':>14s} {'staged rd/out':>14s} "
        f"{'global [ms]':>13s} {'shared [ms]':>13s} {'speedup':>9s} {'|G-S|max':>11s}"
    )
    print("-" * 105)

    for radius in radii:
        row = benchmark_radius(
            a,
            radius,
            block_size=block_size,
            warmup=warmup,
            repetitions=repetitions,
            launches_per_sample=launches_per_sample,
            kernels=kernels,
        )
        rows.append(row)
        print(
            f"{row.radius:4d} {row.reuse:7d} "
            f"{row.ideal_global_reads_per_output:14.2f} "
            f"{row.staged_global_reads_per_output:14.3f} "
            f"{row.global_ms:13.6f} {row.shared_ms:13.6f} "
            f"{row.speedup:9.4f} {row.max_abs_difference:11.3e}"
        )

    print(
        "\nInterpretation: increasing R increases the amount of useful reuse inside "
        "a block. Explicit shared memory can therefore become more attractive, "
        "but the crossover is hardware dependent because L1/L2 caches already "
        "reuse many neighboring global-memory loads. Measure rather than assume."
    )
    return rows


In [ ]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--n", type=int, default=2**22)
    parser.add_argument("--radii", type=int, nargs="+", default=[1, 2, 4, 8, 16, 32])
    parser.add_argument("--block-size", type=int, default=256)
    parser.add_argument("--warmup", type=int, default=3)
    parser.add_argument("--repetitions", type=int, default=7)
    parser.add_argument("--launches", type=int, default=10)
    return parser


In [ ]:
def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    benchmark_reuse(
        n=args.n,
        radii=tuple(args.radii),
        block_size=args.block_size,
        warmup=args.warmup,
        repetitions=args.repetitions,
        launches_per_sample=args.launches,
    )
    return 0


## Baseline radius sweep

This first campaign keeps a large array and explores moderate stencil radii.
It is useful for observing the low-to-moderate data-reuse regime.


In [ ]:
main([
    "--n", "4194304",
    "--radii", "1", "2", "4", "8", "16", "32",
    "--block-size", "256",
    "--warmup", "2",
    "--repetitions", "5",
])



Moving sum: direct global memory vs shared-memory tiling
speedup = T_global / T_shared; values > 1 favor shared memory
reuse = 2R+1 is the number of neighboring outputs using an interior input

   R   reuse  global rd/out  staged rd/out   global [ms]   shared [ms]   speedup    |G-S|max
---------------------------------------------------------------------------------------------------------
   1       3           3.00          1.008      0.219274      0.248371    0.8828   0.000e+00
   2       5           5.00          1.016      0.229958      0.274611    0.8374   0.000e+00
   4       9           9.00          1.031      0.259690      0.300835    0.8632   0.000e+00
   8      17          17.00          1.062      0.338739      0.365603    0.9265   0.000e+00
  16      33          33.00          1.125      0.496842      0.505850    0.9822   0.000e+00
  32      65          65.00          1.250      0.896182      0.829853    1.0799   0.000e+00

Interpretation: increasing R increases the amou

0

## Extended-radius sweep

For larger radii, each input sample is reused by many more neighboring outputs.
The theoretical reuse factor is approximately $2R+1$, so this is the regime
in which explicit shared-memory staging has the best chance to amortize its
loading and synchronization overhead.

The number of grid points is reduced here because the arithmetic work per output
also grows as $2R+1$.  The comparison between global and shared memory remains
fair at each radius because both kernels perform exactly the same numerical work.

The requested radii remain comfortably within the dynamic shared-memory footprint
for a 256-thread block on typical Colab GPUs:

$
M_{\rm sh}=4\,(B+2R)\ \text{bytes}
$

for single-precision data.


In [ ]:
# Wider stencils: stronger data reuse, smaller N to keep runtime practical.
main([
    "--n", "1048576",
    "--radii", "64", "128", "256", "512", "1024",
    "--block-size", "256",
    "--warmup", "2",
    "--repetitions", "3",
])



Moving sum: direct global memory vs shared-memory tiling
speedup = T_global / T_shared; values > 1 favor shared memory
reuse = 2R+1 is the number of neighboring outputs using an interior input

   R   reuse  global rd/out  staged rd/out   global [ms]   shared [ms]   speedup    |G-S|max
---------------------------------------------------------------------------------------------------------
  64     129         129.00          1.500      0.387926      0.345130    1.1240   0.000e+00
 128     257         257.00          2.000      0.765712      0.661133    1.1582   0.000e+00
 256     513         513.00          3.000      1.533549      1.295766    1.1835   0.000e+00
 512    1025        1025.00          5.000      1.635926      1.317664    1.2415   0.000e+00
1024    2049        2049.00          9.000      3.168800      2.540787    1.2472   0.000e+00

Interpretation: increasing R increases the amount of useful reuse inside a block. Explicit shared memory can therefore become more attractiv

0

### How to interpret the extended sweep

A growing shared-memory speedup with increasing $R$ would support the expected
data-reuse argument.  However, the speedup need not increase monotonically forever:
very wide tiles consume more shared memory per block and may reduce occupancy.
Moreover, modern GPU caches can already capture part of the reuse in the direct
global-memory kernel.  Therefore the measured crossover, if any, is hardware
dependent and should be treated as an empirical result rather than a universal
threshold.
